<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-01-logica/04_Logica_Proposicional_Conectivos_e_Permissivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Célula 1: Operadores Lógicos Fundamentais

In [1]:
from typing import Dict
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")

Operadores lógicos proposicionais carregados com sucesso.


Célula 2: Lógica de Decolagem e Intertravamento (Trip)

In [2]:
# Bloco Lógico de Permissivo de Decolagem do Drone
def permissivo_decolagem_drone(gps_ok: bool, bat_low: bool, wind_high: bool, e_stop: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    # Condição de modo exclusivo (Auto XOR Manual)
    modo_valido = XOR(auto_mode, manual_mode)

    # Condição combinada de permissivo
    permissivo = (gps_ok and
                  NOT(bat_low) and
                  NOT(wind_high) and
                  NOT(e_stop) and
                  modo_valido)

    # Condição de trip imediato (RTL - Return to Launch)
    trip = NOT(gps_ok) or bat_low or wind_high or e_stop

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

# Teste com diferentes cenários de voo
cenarios = [
    {"cenario": "Condições Ideais (Auto)", "args": (True, False, False, False, True, False)},
    {"cenario": "Perda de Sinal GPS", "args": (False, False, False, False, True, False)},
    {"cenario": "Bateria Crítica", "args": (True, True, False, False, True, False)},
    {"cenario": "Rajada de Vento (Wind High)", "args": (True, False, True, False, True, False)},
    {"cenario": "Botão de Emergência Acionado", "args": (True, False, False, True, True, False)},
    {"cenario": "Conflito de Modo (Auto e Manual juntos)", "args": (True, False, False, False, True, True)},
]

resultados = []
for c in cenarios:
    res = permissivo_decolagem_drone(*c["args"])
    resultados.append({
        "Cenário": c["cenario"],
        "Permissivo (Decolar)": res["Permissivo_Habilitado"],
        "Trip (Abortar/RTL)": res["Trip_Ativo"],
        "Modo Válido": res["Modo_Valido"]
    })

pd.DataFrame(resultados)

,Cenário,Permissivo (Decolar),Trip (Abortar/RTL),Modo Válido
0,Condições Ideais (Auto),True,False,True
1,Perda de Sinal GPS,False,True,True
2,Bateria Crítica,False,True,True
3,Rajada de Vento (Wind High),False,True,True
4,Botão de Emergência Acionado,False,True,True
5,Conflito de Modo (Auto e Manual juntos),False,False,False


Célula 3: Validação da Tabela-Verdade (Decolagem)

In [3]:
# Geração Automática de Tabela-Verdade para Validação Exaustiva
variaveis = ['gps_ok', 'bat_low', 'wind_high', 'e_stop']
tabela = []

for combo in itertools.product([False, True], repeat=len(variaveis)):
    st = dict(zip(variaveis, combo))
    # Testando com o modo_auto = True e modo_manual = False como padrão
    res = permissivo_decolagem_drone(st['gps_ok'], st['bat_low'], st['wind_high'], st['e_stop'], True, False)
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Ativo']}
    tabela.append(row)

df_tv = pd.DataFrame(tabela)
print(f"Total de combinações avaliadas: {len(df_tv)}")
print(f"Combinações seguras que liberam o drone: {df_tv['Permissivo'].sum()}")
df_tv.head(8)

Total de combinações avaliadas: 16
Combinações seguras que liberam o drone: 1


,gps_ok,bat_low,wind_high,e_stop,Permissivo,Trip
0,False,False,False,False,False,True
1,False,False,False,True,False,True
2,False,False,True,False,False,True
3,False,False,True,True,False,True
4,False,True,False,False,False,True
5,False,True,False,True,False,True
6,False,True,True,False,False,True
7,False,True,True,True,False,True


Célula 4: Lógica de Pulverização (Bomba de Calda)

In [4]:
def permissivo_pulverizacao(is_flying: bool, alt_ok: bool, tank_empty: bool) -> Dict[str, bool]:
    # Permissivo: Voando AND Altitude OK AND Tanque NÃO vazio
    permissivo = is_flying and alt_ok and NOT(tank_empty)

    # Trip: Aborta a pulverização se parar de voar, sair da altitude ou tanque esvaziar
    trip = NOT(is_flying) or NOT(alt_ok) or tank_empty

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip
    }

# Teste Rápido dos Cenários da Bomba
cenarios_bomba = [
    {"cenario": "Pulverização Ideal", "args": (True, True, False)},
    {"cenario": "Drone no Chão (Não Voando)", "args": (False, True, False)},
    {"cenario": "Altitude Incorreta (Risco de Deriva)", "args": (True, False, False)},
    {"cenario": "Tanque Vazio", "args": (True, True, True)},
]

resultados_bomba = []
for c in cenarios_bomba:
    res = permissivo_pulverizacao(*c["args"])
    resultados_bomba.append({
        "Cenário": c["cenario"],
        "Permissivo (Ligar Bomba)": res["Permissivo_Habilitado"],
        "Trip (Cortar Bomba)": res["Trip_Ativo"]
    })

pd.DataFrame(resultados_bomba)

,Cenário,Permissivo (Ligar Bomba),Trip (Cortar Bomba)
0,Pulverização Ideal,True,False
1,Drone no Chão (Não Voando),False,True
2,Altitude Incorreta (Risco de Deriva),False,True
3,Tanque Vazio,False,True
